In [1]:
#import libraies
import torch
import torch.nn as nn  #Neural Network module in PyTorch  It contains all building blocks for neural networks like
import torch.optim as optim  #It helps to update model weights during training
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import os

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [3]:
#Data Preprocessing
transform = transforms.Compose([
    transforms.Resize((224, 224)),   # EfficientNet-B0 input size
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

In [4]:
train_dataset = datasets.ImageFolder(root="dataset/train", transform=transform)
val_dataset = datasets.ImageFolder(root="dataset/val", transform=transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

class_names = train_dataset.classes
print("Classes:", class_names)

Classes: ['dry pitch', 'dusty pitch', 'green pitch']


In [5]:
#model 
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

weights = EfficientNet_B0_Weights.DEFAULT
model = efficientnet_b0(weights=weights)

# Freeze feature layers (optional but recommended)
for param in model.features.parameters():
    param.requires_grad = False

# Replace classifier (for pitch classes)
num_classes = len(class_names)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)

model = model.to(device)

In [6]:
#loss function + optimizer 
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)

In [7]:
#train
epochs = 10

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        # Forward Propagation
        outputs = model(images)

        # Loss Calculation
        loss = criterion(outputs, labels)

        # Backward Propagation
        optimizer.zero_grad()
        loss.backward()

        # Weight Update
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss/len(train_loader):.4f}")

Epoch [1/10], Loss: 1.2196
Epoch [2/10], Loss: 1.0145
Epoch [3/10], Loss: 0.8905
Epoch [4/10], Loss: 0.7981
Epoch [5/10], Loss: 0.6299
Epoch [6/10], Loss: 0.6011
Epoch [7/10], Loss: 0.4634
Epoch [8/10], Loss: 0.4273
Epoch [9/10], Loss: 0.3655
Epoch [10/10], Loss: 0.3232


In [9]:
# Set model to evaluation mode
model.eval()

# Initialize counters
correct = 0
total = 0

# Disable gradient calculation (faster + saves memory)
with torch.no_grad():

    # Loop through validation data
    for images, labels in val_loader:
        
        # Move data to device (CPU/GPU)
        images = images.to(device)
        labels = labels.to(device)

        # Forward pass
        outputs = model(images)

        # Get predicted class (index of max value)
        _, predicted = torch.max(outputs, 1)

        # Update total samples
        total += labels.size(0)

        # Count correct predictions
        correct += (predicted == labels).sum().item()

# Calculate accuracy
accuracy = 100 * correct / total

# Print result
print(f"Validation Accuracy: {accuracy:.2f}%")

Validation Accuracy: 100.00%


In [12]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        correct += (predicted == labels).sum().item()
        total += labels.size(0)

accuracy = correct / total

print("Accuracy:", accuracy)

Accuracy: 1.0


In [ ]:
#model saved 
torch.save(model.state_dict(), "cricket_pitch_model.pth")
print("Model saved successfully!")